# Predict a race from its grid

Type in the starting grid of a race and get the model's predicted finishing order. It uses the 3-feature model from Part 3 (grid position, team form, driver form), retrained on every race from 2010 up to the latest one in the data.

How good is it? In testing on 2023–2026 (Part 3), the predicted order was about 2.25 places off per finisher, exactly right for about 1 finisher in 5, and within 1 place for about half. It can't tell who will retire. It mostly agrees with the grid. Where it disagrees, it's usually moving a fast car that qualified badly up the order. It's a race-watching companion, not a betting tool: bookmakers' odds already include far more information than this model has.

Before using it for a new race, download the latest data from Kaggle (see the README), replace `data/raw/`, and re-run `01_build_dataset.ipynb`, so the form numbers include the most recent races.

In [1]:
import unicodedata
import difflib
import pandas as pd
from sklearn.linear_model import LinearRegression

df = pd.read_csv("../data/processed/results_joined.csv", low_memory=False)

# Put every result in time order (same as 3.2; the date comes from 01)
data = df.sort_values(["date", "positionOrder"]).reset_index(drop=True)

last = data[data["date"] == data["date"].max()].iloc[0]
print(f"Data runs up to the {last['year']} {last['race_name']} ({last['date']})")

Data runs up to the 2026 Mexico City Grand Prix (2026-11-01)


In [2]:
# The same form features as 3.2: average finish over the previous 5 races
# (shift(1) leaves out the race itself, so the model only learns from what was known beforehand)
data["driver_form"] = data.groupby("driverId")["positionOrder"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())

team_races = data.groupby(["constructorId", "raceId", "date"])["positionOrder"].mean().reset_index()
team_races = team_races.sort_values("date")

# Same as 3.2: a reused team ID starts its form again after more than a year without racing
gap_days = pd.to_datetime(team_races["date"]).groupby(team_races["constructorId"]).diff().dt.days
team_races["spell"] = (gap_days > 365).groupby(team_races["constructorId"]).cumsum()
team_races["team_form"] = team_races.groupby(["constructorId", "spell"])["positionOrder"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())
data = data.merge(team_races[["constructorId", "raceId", "team_form"]], on=["constructorId", "raceId"], how="left")

features = ["grid", "team_form", "driver_form"]  # the 3 features that mattered in 3.5

In [3]:
def fill_gaps(table):
    # Same gap filling as 3.2: a rookie gets the team's level, a new team gets the driver's level,
    # and with neither, the grid spot is used
    table["driver_form"] = table["driver_form"].fillna(table["team_form"])
    table["team_form"] = table["team_form"].fillna(table["driver_form"])
    table["driver_form"] = table["driver_form"].fillna(table["grid"])
    table["team_form"] = table["team_form"].fillna(table["grid"])
    return table


def train_model(history):
    # Train on every finisher from 2010 on who started from a real grid spot, using only `history`
    rows = fill_gaps(history[(history["year"] >= 2010) & (history["grid"] > 0) & history["started"]].copy())
    finishers = rows[rows["finished"]]
    return LinearRegression().fit(finishers[features], finishers["position"])


def simplify(text):
    # Lowercase and drop accents, so "hulkenberg" matches "Hülkenberg"
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().lower().strip()


def find_driver(name, history):
    # A full name always works. A surname works if it fits exactly one driver from the last 3 seasons
    if name in set(history["driver"]):
        return [name]
    recent = history[history["year"] >= history["year"].max() - 2]
    return sorted({d for d, s in zip(recent["driver"], recent["surname"])
                   if simplify(name) in (simplify(d), simplify(s))})

In [4]:
def predict_race(grid_order, teams=None, before=None):
    """
    grid_order: driver names in starting order, pole sitter first (pit-lane starters at the end).
    teams:      optional {driver: team} for a rookie, a driver who has changed team, or a new team.
                Without it, each driver gets the team from their most recent race, which can be out of date.
    before:     optional date like "2026-11-01" to replay a past race using only earlier races.
    """
    if not grid_order:
        print("The grid is empty.")
        return None
    teams = teams or {}
    if before is not None:
        # Dates are stored as text like "2026-11-01", so write `before` the same way.
        # Otherwise "2026-11-1" would sort after "2026-11-01" and let that race's result in
        before = pd.Timestamp(before).strftime("%Y-%m-%d")
        history = data[data["date"] < before]
        team_history = team_races[team_races["date"] < before]
    else:
        history, team_history = data, team_races
    model = train_model(history)
    used_teams = set()
    race_year = pd.Timestamp(before).year if before is not None else history["year"].max()
    guessed = []  # teams taken from a race in an earlier season, which may be out of date

    rows, problems = [], []
    for grid_spot, name in enumerate(grid_order, start=1):
        matches = find_driver(name, history)
        team = teams.get(name)
        if len(matches) > 1:
            problems.append(f"'{name}' fits more than one driver: {matches}. Use the full name.")
            continue
        if not matches and team is None:
            # Look for near-misses among recent full names and surnames (typos like "Leclrec")
            recent = history[history["year"] >= history["year"].max() - 2]
            spellings = {simplify(d): d for d in recent["driver"]} | {simplify(s): d for d, s in zip(recent["driver"], recent["surname"])}
            close = sorted({spellings[k] for k in difflib.get_close_matches(simplify(name), spellings, n=3, cutoff=0.7)})
            hint = f" Did you mean {close}?" if close else ""
            problems.append(f"'{name}' isn't in the data.{hint} "
                            f"If they're a rookie, add their team: teams={{'{name}': 'Team name'}}")
            continue
        driver = matches[0] if matches else name
        past = history[history["driver"] == driver]

        # Team: the one given in `teams` (as typed in the grid, or the full name), otherwise their most recent team
        key = name if name in teams else driver
        team = teams.get(key)
        if team is None:
            team = past["team"].iloc[-1]
            if past["year"].iloc[-1] < race_year:
                guessed.append(f"{driver}: {team} (last raced in {past['year'].iloc[-1]})")
        else:
            used_teams.add(key)
        if team in set(history["team"]):
            team_id = history.loc[history["team"] == team, "constructorId"].iloc[-1]
            team_past = team_history[team_history["constructorId"] == team_id]
            team_past = team_past[team_past["spell"] == team_past["spell"].iloc[-1]]  # its latest spell only
            team_form = team_past["positionOrder"].tail(5).mean()
        else:
            # A new team has no form yet, so fill_gaps uses the driver's form instead (same as 3.2)
            recent_teams = sorted(history.loc[history["year"] == history["year"].max(), "team"].unique())
            spellings = {simplify(t): t for t in recent_teams}
            close = [spellings[k] for k in difflib.get_close_matches(simplify(team), spellings, n=3, cutoff=0.8)]
            hint = f" If that's a typo, did you mean {close}?" if close else ""
            print(f"Note: team '{team}' isn't in the data yet, so {driver}'s own form stands in for it.{hint}")
            team_form = float("nan")

        rows.append({
            "driver": driver,
            "team": team,
            "grid": grid_spot,
            "driver_form": past["positionOrder"].tail(5).mean(),        # NaN for a rookie
            "team_form": team_form,
        })

    if guessed:
        print("Note: these teams come from each driver's last race in an earlier season. "
              "If anyone has moved, pass teams={'Driver': 'Team'}:")
        for g in guessed:
            print(" -", g)
    unused = sorted(set(teams) - used_teams)
    if unused:
        print(f"Note: these names in `teams` don't match anyone in the grid, so they were ignored: {unused}")

    drivers = [r["driver"] for r in rows]
    repeated = sorted({d for d in drivers if drivers.count(d) > 1})
    if repeated:
        problems.append(f"Listed more than once: {repeated}")
    if problems:
        print("Please fix the grid first:")
        for p in problems:
            print(" -", p)
        return None

    table = fill_gaps(pd.DataFrame(rows))
    table["predicted"] = model.predict(table[features])
    table = table.sort_values("predicted").reset_index(drop=True)
    table.index = table.index + 1                      # 1 = predicted winner
    table.index.name = "predicted_finish"
    table["vs_grid"] = table["grid"] - table.index     # positive = expected to gain places
    return table[["driver", "team", "grid", "vs_grid", "driver_form", "team_form"]].round(1)

## Example: replay the last race in the data

`before=` hides that race and everything after it, so this is an honest test: the model only knows what was known before the race started.

In [5]:
last_race = data[data["date"] == data["date"].max()]
# Grid order, with any pit-lane starters (grid 0) at the end, as predict_race expects
last_grid = (last_race[last_race["grid"] > 0].sort_values("grid")["driver"].tolist()
             + last_race[last_race["grid"] == 0]["driver"].tolist())

# Teams are known before a race, so pass them in. This also covers a driver making their debut
last_teams = last_race.set_index("driver")["team"].to_dict()
replay = predict_race(last_grid, teams=last_teams, before=last_race["date"].iloc[0])

# Add what really happened
replay["actual_finish"] = replay["driver"].map(last_race.set_index("driver")["positionOrder"])
replay["status"] = replay["driver"].map(last_race.set_index("driver")["status"])
replay

,driver,team,grid,vs_grid,driver_form,team_form,actual_finish,status
predicted_finish,,,,,,,,
1,Max Verstappen,Red Bull,5,4,1.8,3.9,1,Finished
2,Charles Leclerc,Ferrari,3,1,3.2,5.7,4,Finished
3,Lewis Hamilton,Ferrari,1,-2,8.2,5.7,2,Finished
4,Andrea Kimi Antonelli,Mercedes,4,0,2.4,5.5,3,Finished
5,George Russell,Mercedes,2,-3,8.6,5.5,6,Finished
6,Isack Hadjar,Red Bull,7,1,6.0,3.9,9,Finished
7,Lando Norris,McLaren,6,-1,8.0,7.9,5,Finished
8,Arvid Lindblad,RB F1 Team,8,0,10.6,10.2,18,Retired
9,Oscar Piastri,McLaren,12,3,7.8,7.9,13,Finished


## Your race

1. Run the cell below to see how the data spells each driver's name. Surnames are fine too (for example `"Hulkenberg"`).
2. Type the grid into `my_grid`, pole sitter first. Drivers starting from the pit lane go at the end.
3. Run the cell after it. If a rookie isn't in the data yet, add their team, e.g. `predict_race(my_grid, teams={"New Driver": "Williams"})`. The same works for a driver who has changed team, or a brand-new team (its form is then taken from the driver's). Without `teams`, each driver gets the team from their most recent race, so at the start of a season check for drivers who moved over the winter.

In [6]:
# Drivers and teams from the most recent race
print(last_race.sort_values("team")[["driver", "team"]].to_string(index=False))

               driver             team
     Franco Colapinto   Alpine F1 Team
         Pierre Gasly   Alpine F1 Team
      Fernando Alonso     Aston Martin
         Lance Stroll     Aston Martin
      Nico Hülkenberg             Audi
    Gabriel Bortoleto             Audi
         Sergio Pérez Cadillac F1 Team
      Valtteri Bottas Cadillac F1 Team
      Charles Leclerc          Ferrari
       Lewis Hamilton          Ferrari
         Ryo Hirakawa     Haas F1 Team
       Oliver Bearman     Haas F1 Team
         Lando Norris          McLaren
        Oscar Piastri          McLaren
       George Russell         Mercedes
Andrea Kimi Antonelli         Mercedes
          Liam Lawson       RB F1 Team
       Arvid Lindblad       RB F1 Team
       Max Verstappen         Red Bull
         Isack Hadjar         Red Bull
         Carlos Sainz         Williams
      Alexander Albon         Williams


In [7]:
my_grid = [
    # "Verstappen",
    # "Hamilton",
    # ...
]

if my_grid:
    display(predict_race(my_grid))
else:
    print("Type the grid into my_grid above, then run this cell again.")

Type the grid into my_grid above, then run this cell again.
